# Burnout Risk Indicator Analysis

## Project Overview
Analyze workload, leave, hours, survey, and performance signals to explore possible burnout patterns across teams.

## Learning Objectives
- Build a composite burnout risk indicator from multiple signals
- Identify high-risk teams and employee segments
- Analyze the relationship between workload, leave patterns, and engagement
- Detect early warning signs of burnout

## Problem Statement
HR and people managers need data-driven insights into which employees and teams may be at risk of burnout, so they can intervene proactively before performance and health decline.

## Why This Project Matters
Burnout leads to increased attrition, decreased productivity, health issues, and workplace disengagement. Early detection through data patterns enables preventive action.

## Dataset Overview
Synthetic employee wellness dataset: ~3,500 employees with workload metrics, leave usage, survey scores, and performance data.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 3500
departments = np.random.choice(['Engineering', 'Sales', 'Operations', 'Support', 'Marketing', 'Finance'],
                                 n, p=[0.24, 0.20, 0.16, 0.15, 0.13, 0.12])
levels = np.random.choice(['Junior', 'Mid', 'Senior', 'Lead'], n, p=[0.25, 0.32, 0.27, 0.16])
tenure = np.clip(np.random.exponential(4, n), 0.5, 18).round(1)

avg_weekly_hours = np.clip(np.random.normal(44, 6, n), 30, 70).round(1)
weekend_work_pct = np.clip(np.random.beta(2, 8, n) * 100, 0, 80).round(0)
after_hours_msgs = np.clip(np.random.poisson(8, n), 0, 50)
leave_days_used = np.clip(np.random.normal(12, 5, n), 0, 25).round(0)
leave_days_entitled = 20
leave_utilization = (leave_days_used / leave_days_entitled * 100).round(1)
consecutive_no_leave_weeks = np.clip(np.random.exponential(6, n), 0, 30).round(0)

wellbeing_score = np.clip(np.random.normal(3.4, 0.9, n), 1.0, 5.0).round(1)
engagement_score = np.clip(np.random.normal(3.5, 0.8, n), 1.0, 5.0).round(1)
perf_trend = np.random.choice(['Improving', 'Stable', 'Declining'], n, p=[0.20, 0.55, 0.25])

# Burnout risk composite
from sklearn.preprocessing import MinMaxScaler
risk_features = np.column_stack([
    avg_weekly_hours, weekend_work_pct, after_hours_msgs,
    100 - leave_utilization, consecutive_no_leave_weeks,
    5 - wellbeing_score, 5 - engagement_score,
    (perf_trend == 'Declining').astype(float) * 100
])
scaler = MinMaxScaler()
scaled = scaler.fit_transform(risk_features)
burnout_score = (scaled.mean(axis=1) * 100).round(1)

df = pd.DataFrame({
    'EmployeeID': [f'E{i:05d}' for i in range(n)],
    'Department': departments, 'Level': levels, 'TenureYears': tenure,
    'AvgWeeklyHours': avg_weekly_hours, 'WeekendWorkPct': weekend_work_pct,
    'AfterHoursMsgs': after_hours_msgs, 'LeaveDaysUsed': leave_days_used,
    'LeaveUtilization': leave_utilization, 'ConsecNoLeaveWeeks': consecutive_no_leave_weeks,
    'WellbeingScore': wellbeing_score, 'EngagementScore': engagement_score,
    'PerfTrend': perf_trend, 'BurnoutRiskScore': burnout_score,
})
df['RiskCategory'] = pd.cut(df['BurnoutRiskScore'], bins=[0, 30, 50, 70, 100],
                              labels=['Low', 'Moderate', 'High', 'Critical'])

print(f'Dataset: {df.shape}')
print(f'\nBurnout Risk Category distribution:\n{df["RiskCategory"].value_counts().sort_index()}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nBurnout Risk Score stats:\n{df["BurnoutRiskScore"].describe().round(1)}')

## Burnout Risk Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['BurnoutRiskScore'], bins=30, edgecolor='black', color='coral', alpha=0.7)
axes[0].axvline(50, color='orange', linestyle='--', label='Moderate threshold')
axes[0].axvline(70, color='red', linestyle='--', label='High threshold')
axes[0].set_title('Burnout Risk Score Distribution')
axes[0].legend()

df['RiskCategory'].value_counts().reindex(['Low','Moderate','High','Critical']).plot.bar(
    ax=axes[1], edgecolor='black', color=['green', 'gold', 'orange', 'red'])
axes[1].set_title('Employees by Risk Category')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'risk_distribution.png'), dpi=100, bbox_inches='tight')
plt.show()

## Risk by Department and Level

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dept_risk = df.groupby('Department')['BurnoutRiskScore'].mean().sort_values()
dept_risk.plot.barh(ax=axes[0], edgecolor='black', color='coral')
axes[0].set_title('Avg Burnout Risk by Department')

level_order = ['Junior', 'Mid', 'Senior', 'Lead']
level_risk = df.groupby('Level')['BurnoutRiskScore'].mean().reindex(level_order)
level_risk.plot.bar(ax=axes[1], edgecolor='black', color='steelblue')
axes[1].set_title('Avg Burnout Risk by Level')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'risk_dept_level.png'), dpi=100, bbox_inches='tight')
plt.show()

# % in High/Critical by department
high_risk = df[df['RiskCategory'].isin(['High', 'Critical'])]
dept_high = high_risk.groupby('Department').size() / df.groupby('Department').size() * 100
print('% High/Critical Risk by Department:')
print(dept_high.sort_values(ascending=False).round(1))

## Workload Indicators

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for cat, color in zip(['Low', 'Moderate', 'High', 'Critical'], ['green', 'gold', 'orange', 'red']):
    sub = df[df['RiskCategory'] == cat]
    axes[0,0].hist(sub['AvgWeeklyHours'], bins=20, alpha=0.4, label=cat, color=color, edgecolor='black')
axes[0,0].set_title('Weekly Hours by Risk Category')
axes[0,0].legend()

axes[0,1].scatter(df['AvgWeeklyHours'], df['BurnoutRiskScore'], alpha=0.1, s=5, color='steelblue')
axes[0,1].set_xlabel('Avg Weekly Hours')
axes[0,1].set_ylabel('Burnout Risk Score')
axes[0,1].set_title('Hours vs Burnout Risk')

axes[1,0].scatter(df['LeaveUtilization'], df['BurnoutRiskScore'], alpha=0.1, s=5, color='teal')
axes[1,0].set_xlabel('Leave Utilization %')
axes[1,0].set_ylabel('Burnout Risk Score')
axes[1,0].set_title('Leave Usage vs Burnout Risk')

risk_by_trend = df.groupby('PerfTrend')['BurnoutRiskScore'].mean().reindex(['Improving', 'Stable', 'Declining'])
risk_by_trend.plot.bar(ax=axes[1,1], edgecolor='black', color='goldenrod')
axes[1,1].set_title('Burnout Risk by Performance Trend')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'workload_indicators.png'), dpi=100, bbox_inches='tight')
plt.show()

## Factor Contribution to Burnout Risk

In [ ]:
risk_corr = df[['AvgWeeklyHours', 'WeekendWorkPct', 'AfterHoursMsgs',
                 'LeaveUtilization', 'ConsecNoLeaveWeeks',
                 'WellbeingScore', 'EngagementScore', 'BurnoutRiskScore']].corr()['BurnoutRiskScore'].drop('BurnoutRiskScore')
print('Factor Correlations with Burnout Risk Score:')
print(risk_corr.sort_values(ascending=False).round(3))

fig, ax = plt.subplots(figsize=(10, 5))
risk_corr.abs().sort_values().plot.barh(ax=ax, edgecolor='black', color='coral')
ax.set_title('Factor Correlation Strength with Burnout Risk')
ax.set_xlabel('|Correlation|')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'factor_contribution.png'), dpi=100, bbox_inches='tight')
plt.show()

## High-Risk Employee Profiles

In [ ]:
critical = df[df['RiskCategory'] == 'Critical']
low = df[df['RiskCategory'] == 'Low']
profile_cols = ['AvgWeeklyHours', 'WeekendWorkPct', 'AfterHoursMsgs',
                'LeaveUtilization', 'ConsecNoLeaveWeeks', 'WellbeingScore', 'EngagementScore']
comp = pd.DataFrame({
    'Critical Risk': critical[profile_cols].mean(),
    'Low Risk': low[profile_cols].mean(),
}).round(1)
print('Profile Comparison — Critical vs Low Risk:')
print(comp)

## Interpretation and Business Insight
- **Excessive hours and weekend work** are the strongest workload indicators of burnout risk
- **Low leave utilization** is a key warning sign — employees not taking leave are accumulating stress
- **Declining performance** often accompanies high burnout scores — it's both a cause and a consequence
- **Wellbeing and engagement scores** inversely correlate with burnout — survey data is a valid early warning
- **After-hours messaging** captures "always-on" culture effects even when official hours look normal
- **Support and Operations** teams tend to have higher burnout risk — frontline roles face relentless demand
- **Senior and Lead** levels show elevated risk — individual contributor stress plus leadership burden

## Limitations
- Synthetic data with pre-built risk scoring
- No clinical burnout assessment (Maslach Burnout Inventory)
- Self-reported survey data may be biased
- No manager behavior or culture data
- Burnout is complex and context-dependent — no single score captures it fully

## How to Improve This Project
- Validate against clinical burnout assessments
- Add longitudinal tracking (burnout trajectory over time)
- Include manager 1-on-1 frequency and quality data
- Add workload variability (sustained high vs occasional peaks)
- Build predictive models for attrition risk from burnout signals

## Production Considerations
- Monthly burnout risk dashboards for people managers
- Automated alerts for employees crossing risk thresholds
- Team-level burnout heatmaps for leadership review
- Integration with calendar/messaging analytics for objective workload data
- Privacy-preserving aggregation (team-level, not individual naming)

## Common Mistakes
- Using only hours worked as a burnout proxy (burnout is multifactorial)
- Ignoring leave patterns — not taking leave is a stronger signal than working long hours
- Blaming individuals instead of addressing systemic workload issues
- Not acting on data — dashboards without intervention are useless

## Mini Challenge / Exercises
1. What % of employees with >50 weekly hours AND <50% leave utilization are in High/Critical risk?
2. Which department × level combination has the highest average burnout risk?
3. Calculate the estimated attrition cost if 30% of Critical-risk employees leave (assume $80K replacement cost).
4. Design a simple intervention scorecard: for each risk factor, suggest one concrete action HR can take.

## Final Summary / Key Takeaways
- Burnout risk analysis combines workload, leave, survey, and performance signals
- No single metric captures burnout — composite indicators are more reliable
- Early warning systems enable proactive intervention before performance collapse
- Team-level patterns often reveal systemic issues (understaffing, poor boundaries)
- Organizations that monitor and act on burnout data retain more talent and maintain higher productivity